In [0]:
import gc
from pyspark.sql.functions import col, when, count, lit
from pyspark.ml.feature import VectorAssembler, FeatureHasher
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

# 1. DATA PREP
print("Preparing Data...")
raw_df = spark.table("workspace.capstone_project.toronto_model_ready") \
    .filter(col("delay_indicator").isNotNull()) \
    .select(
        "delay_indicator", "incident_category", "season", "unified_call_source", 
        "location_area", "hour", "day_of_week", "month", "year", 
        "unified_alarm_level", "calls_past_30min", "calls_past_60min"
    )

# 2. WEIGHT CALCULATIONS
label_counts = raw_df.groupBy("delay_indicator").count().collect()
total_rows = sum(r['count'] for r in label_counts)
w_map = {r['delay_indicator']: total_rows / (2.0 * r['count']) for r in label_counts}

df = raw_df.withColumn("weight", 
    when(col("delay_indicator") == 1, w_map[1]).otherwise(w_map[0]))

# 3. PIPELINE COMPONENTS
hasher = FeatureHasher(inputCols=['incident_category', 'season', 'unified_call_source', 'location_area'],
                       outputCol="categorical_features", numFeatures=512) 

numeric_cols = ['hour', 'day_of_week', 'month', 'year', 'unified_alarm_level', 'calls_past_30min', 'calls_past_60min']
assembler = VectorAssembler(inputCols=numeric_cols + ["categorical_features"], outputCol="features")

# Split data
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# 4. MODELS 
models = {
    "Random Forest": RandomForestClassifier(
        labelCol="delay_indicator", featuresCol="features", weightCol="weight",
        numTrees=60, maxDepth=8, maxBins=32, seed=42),
    
    "GBT Classifier": GBTClassifier(
        labelCol="delay_indicator", featuresCol="features",
        maxIter=25, maxDepth=4, stepSize=0.1, seed=42)
}

# 5. EXECUTION
results = []
evaluator = BinaryClassificationEvaluator(labelCol="delay_indicator", metricName="areaUnderPR")

for name, clf in models.items():
    print(f"Training {name} (Lean Mode)...")
    pipeline = Pipeline(stages=[hasher, assembler, clf])
    
    model = pipeline.fit(train_df)
    
    preds = model.transform(test_df)
    pr_auc = evaluator.evaluate(preds)
    
    print(f"{name} PR-AUC: {pr_auc:.4f}")
    results.append((name, pr_auc))
    gc.collect()

print("\nFinal Performance Summary:")
for name, score in results:
    print(f"{name}: {score:.4f}")